# Sistema de Recuperación Semántica Local (RAG)
## Flujo End-to-End con LangChain (LCEL), ChromaDB y Pydantic

Este notebook implementa un sistema RAG local completo que incluye:
1. **Módulo de Ingesta:** Carga de documentos de `/data`, chunking estratégico con `RecursiveCharacterTextSplitter` y persistencia en ChromaDB.
2. **Capa de Recuperación:** Retriever semántico con control de límite de contexto (`top_k = 4`).
3. **Generación Grounded:** Cadena LCEL asíncrona con prompt anti-alucinaciones y salida tipada mediante `PydanticOutputParser`.

In [ ]:
# 1. Configuración de dependencias y variables de entorno
import os
import asyncio
from pathlib import Path
from dotenv import load_dotenv

# Cargar variables de entorno desde .env
load_dotenv()

from config import DATA_DIR, VECTORSTORE_DIR, COLLECTION_NAME, TOP_K, CHUNK_SIZE, CHUNK_OVERLAP
from embeddings import get_embedding_function
from schemas import RAGResponse

print("Variables y módulos cargados correctamente.")

### 2. Ingesta y Fragmentación Estratégica (Chunking)
Leemos los documentos de la plataforma ficticia `Company Cloud` y aplicamos `RecursiveCharacterTextSplitter` basado en tokens.

In [ ]:
from ingest import load_documents_from_directory, split_documents, ingest_data

# Ejecutamos la ingesta hacia ChromaDB local
vectorstore = ingest_data(force_reload=False)
print(f"Colección activa con {vectorstore._collection.count()} fragmentos indexados.")

### 3. Construcción del Prompt Anti-Alucinación y Cadena LCEL
Configuramos el prompt como 'filtro de veracidad' que instruye al LLM a decir 'No lo sé' si la información no está en el contexto.

In [ ]:
from rag_chain import build_rag_chain, get_rag_response

# Instanciamos la cadena LCEL
chain = build_rag_chain()
print("Cadena LCEL construida exitosamente con PydanticOutputParser.")

### 4. Prueba 1: Pregunta cuya respuesta ESTÁ en los documentos
Consultamos sobre el SLA de incidentes Sev-1 y el proceso de escalamiento.

In [ ]:
query_1 = (
    "¿Cuál es el tiempo máximo de primera respuesta (SLA) para un incidente Sev-1 "
    "y a quién se debe escalar si la causa raíz no se identifica en los primeros 20 minutos?"
)

response_1 = await get_rag_response(query_1)

print("=== PRUEBA 1: PREGUNTA VÁLIDA ===")
print(f"Pregunta:   {query_1}")
print(f"Respuesta:  {response_1.answer}")
print(f"Fuentes:    {response_1.sources}")
print(f"Confianza:  {response_1.confidence}")

### 5. Prueba 2: Pregunta Trampa (Información NO presente en los documentos)
Consultamos sobre vacaciones y reembolso de comidas para verificar que el modelo no alucine.

In [ ]:
query_2 = (
    "¿Cuál es el procedimiento para solicitar el reembolso de gastos de almuerzo "
    "y cuántos días de vacaciones anuales le corresponden a un desarrollador senior?"
)

response_2 = await get_rag_response(query_2)

print("=== PRUEBA 2: PREGUNTA TRAMPA ===")
print(f"Pregunta:   {query_2}")
print(f"Respuesta:  {response_2.answer}")
print(f"Fuentes:    {response_2.sources}")
print(f"Confianza:  {response_2.confidence}")